In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [3]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [4]:
import os
import json
import glob
import numpy as np
import pandas as pd

In [5]:
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
DAY20_PATH = os.path.join(BASE_PATH, "day20_outputs")
DAY22_PATH = os.path.join(BASE_PATH, "day22_outputs")
DAY24_PATH = os.path.join(BASE_PATH, "day24_outputs")
DAY25_PATH = os.path.join(BASE_PATH, "day25_outputs")
os.makedirs(DAY25_PATH, exist_ok=True)
print("DETAILED ERROR / FAILURE ANALYSIS")

DETAILED ERROR / FAILURE ANALYSIS


In [7]:
print("\n Checking Day 20 analysis evidence")
if os.path.exists(DAY20_PATH):
    print(" Day 20 output folder found.")
else:
    print("Day 20 output folder not found.")
day20_files = []
if os.path.exists(DAY20_PATH):
    for root, dirs, files in os.walk(DAY20_PATH):
        for file in files:
            day20_files.append(
                os.path.join(root, file))
print("Day 20 files found:", len(day20_files))
for file in day20_files[:20]:
    print(" •", os.path.relpath(file, DAY20_PATH))
if len(day20_files) > 20:
    print(" ")


 Checking Day 20 analysis evidence
 Day 20 output folder found.
Day 20 files found: 12
 • day20_overall_test_results.csv
 • day20_SOC_test_predictions.csv
 • day20_remaining_time_test_predictions.csv
 • day20_robustness_ablation_table.csv
 • day20_error_analysis.csv
 • day20_targetwise_feature_importance.csv
 • day20_SOC_error_distribution.png
 • day20_remaining_time_feature_importance.png
 • day20_SOC_feature_importance.png
 • day20_remaining_time_error_distribution.png
 • day20_research_insight.txt
 • day20_summary.json


In [8]:
print("\n Searching for existing error-analysis files...")
error_keywords = ["error","residual","prediction","pred","actual","analysis","comparison"]
candidate_files = []
for file in day20_files:
    filename = os.path.basename(file).lower()
    if any(keyword in filename for keyword in error_keywords):
        if file.lower().endswith(
            (".csv", ".xlsx", ".json", ".txt")):
            candidate_files.append(file)
print("Potential analysis files found:", len(candidate_files))
for file in candidate_files:
    print(" •", os.path.relpath(file, DAY20_PATH))


 Searching for existing error-analysis files...
Potential analysis files found: 3
 • day20_SOC_test_predictions.csv
 • day20_remaining_time_test_predictions.csv
 • day20_error_analysis.csv


In [9]:
print("\n Reading available Day 20 CSV evidence...")
dataframes = []
for file in candidate_files:
    if not file.lower().endswith(".csv"):
        continue
    try:
        df = pd.read_csv(file)
        if len(df) == 0:
            continue
        dataframes.append((file,df))
        print(" ",os.path.relpath(file, DAY20_PATH),"| rows:",len(df),"| columns:",len(df.columns))
    except Exception as e:
        print(" Could not read:",os.path.basename(file),"|",e)


 Reading available Day 20 CSV evidence...
  day20_SOC_test_predictions.csv | rows: 2250 | columns: 13
  day20_remaining_time_test_predictions.csv | rows: 2286 | columns: 14
  day20_error_analysis.csv | rows: 2 | columns: 7


In [11]:
print("\n Inspecting error-related columns")
error_column_summary = []
for file, df in dataframes:
    for column in df.columns:
        column_lower = str(column).lower()
        if any(
            keyword in column_lower
            for keyword in ["error","residual","absolute","actual","pred"]):
            error_column_summary.append({"file": os.path.basename(file),"column": column,"rows": len(df)})
if error_column_summary:
    error_columns_df = pd.DataFrame(error_column_summary)
    print(error_columns_df.to_string(index=False))
else:
    print("No standard error/prediction columns were automatically identified.")


 Inspecting error-related columns
                                     file                         column  rows
           day20_SOC_test_predictions.csv                  Predicted_SOC  2250
           day20_SOC_test_predictions.csv                      SOC_Error  2250
           day20_SOC_test_predictions.csv             SOC_Absolute_Error  2250
day20_remaining_time_test_predictions.csv       Predicted_Remaining_Time  2286
day20_remaining_time_test_predictions.csv           Remaining_Time_Error  2286
day20_remaining_time_test_predictions.csv  Remaining_Time_Absolute_Error  2286
                 day20_error_analysis.csv                     Mean_Error     2
                 day20_error_analysis.csv            Mean_Absolute_Error     2
                 day20_error_analysis.csv          Median_Absolute_Error     2
                 day20_error_analysis.csv 90th_Percentile_Absolute_Error     2
                 day20_error_analysis.csv         Maximum_Absolute_Error     2


In [12]:
print("\n Calculating available error statistics")
error_statistics = []
for file, df in dataframes:
    for column in df.columns:
        column_lower = str(column).lower()
        if "error" in column_lower or "residual" in column_lower:
            numeric_values = pd.to_numeric(
                df[column],errors="coerce").dropna()
            if len(numeric_values) == 0:
                continue
            abs_values = np.abs(numeric_values)
            error_statistics.append({"File": os.path.basename(file),"Error Column": column,"Count": len(numeric_values),"Mean Error":float(numeric_values.mean()),"Mean Absolute Error":float(abs_values.mean()),"Maximum Absolute Error":float(abs_values.max()),"Median Absolute Error":float(np.median(abs_values)),"90th Percentile Absolute Error":float(np.percentile(abs_values, 90))})
if error_statistics:
    error_statistics_df = pd.DataFrame(error_statistics)
    print(error_statistics_df.to_string(index=False))
else:
    print("No directly usable error column was found.")


 Calculating available error statistics
                                     File                   Error Column  Count  Mean Error  Mean Absolute Error  Maximum Absolute Error  Median Absolute Error  90th Percentile Absolute Error
           day20_SOC_test_predictions.csv                      SOC_Error   2250    0.099500             1.353850               32.558224               0.454127                        2.621832
           day20_SOC_test_predictions.csv             SOC_Absolute_Error   2250    1.353850             1.353850               32.558224               0.454127                        2.621832
day20_remaining_time_test_predictions.csv           Remaining_Time_Error   2286  492.818450           542.106640             3260.411979             437.463137                     1335.971877
day20_remaining_time_test_predictions.csv  Remaining_Time_Absolute_Error   2286  542.106640           542.106640             3260.411979             437.463137                     1335.971877

In [14]:
print("\n Checking for actual-versus-predicted data...")
prediction_pairs = []
for file, df in dataframes:
    actual_columns = [c for c in df.columns if "actual" in str(c).lower()or "true" in str(c).lower()or "target" in str(c).lower()]
    predicted_columns = [c for c in df.columns if "pred" in str(c).lower() or "forecast" in str(c).lower()]
    if actual_columns and predicted_columns:
        print("Prediction evidence found:",os.path.basename(file))
        print("  Actual columns:", actual_columns)
        print("  Predicted columns:", predicted_columns)
        for actual_col in actual_columns:
            for predicted_col in predicted_columns:
                actual = pd.to_numeric(df[actual_col],errors="coerce")
                predicted = pd.to_numeric(df[predicted_col],errors="coerce")
                valid = (actual.notna()& predicted.notna())
                if valid.sum() == 0:
                    continue
                actual_valid = actual[valid]
                predicted_valid = predicted[valid]
                error = (predicted_valid- actual_valid)
                abs_error = np.abs(error)
                prediction_pairs.append({
                    "File":os.path.basename(file),
                    "Actual Column":actual_col,
                    "Predicted Column":predicted_col,
                    "Samples":int(valid.sum()),
                    "MAE":float(abs_error.mean()),
                    "RMSE":float(np.sqrt(np.mean(error ** 2))),
                    "Mean Error":float(error.mean()),
                    "Maximum Absolute Error":float(abs_error.max()),
                    "90th Percentile Absolute Error":
                        float(np.percentile(abs_error,90))})
if prediction_pairs:
    prediction_analysis_df = pd.DataFrame(prediction_pairs)
    print(prediction_analysis_df.to_string(index=False))
else:
    prediction_analysis_df = pd.DataFrame()
    print("No actual-versus-predicted pair was automatically detected.")


 Checking for actual-versus-predicted data...
No actual-versus-predicted pair was automatically detected.


In [16]:
print("\n Identifying high-error cases")
high_error_summary = []
for file, df in dataframes:
    actual_columns = [c for c in df.columns if "actual" in str(c).lower()or "true" in str(c).lower()or "target" in str(c).lower()]
    predicted_columns = [c for c in df.columns if "pred" in str(c).lower()]
    if not actual_columns or not predicted_columns:
        continue
    for actual_col in actual_columns:
        for predicted_col in predicted_columns:
            actual = pd.to_numeric(df[actual_col],errors="coerce")
            predicted = pd.to_numeric(df[predicted_col], errors="coerce")
            valid = (actual.notna()& predicted.notna())
            if valid.sum() == 0:
                continue
            temp = df.loc[valid].copy()
            temp["_actual_numeric"] = actual[valid]
            temp["_predicted_numeric"] = predicted[valid]
            temp["_absolute_error"] = np.abs(temp["_predicted_numeric"]- temp["_actual_numeric"])
            threshold = temp["_absolute_error"].quantile(0.90)
            high_error_cases = temp[temp["_absolute_error"] >= threshold]
            high_error_summary.append({
                "File":os.path.basename(file),
                "Actual": actual_col,
                "Predicted":predicted_col,
                "High-error threshold (90th percentile)":float(threshold),
                "High-error samples":int(len(high_error_cases)),
                "Maximum absolute error":float(temp["_absolute_error"].max())})
if high_error_summary:
    high_error_df = pd.DataFrame(high_error_summary)
    print( high_error_df.to_string(index=False))
else:
    high_error_df = pd.DataFrame()
    print("High-error cases could not be automatically identified.")


 Identifying high-error cases
High-error cases could not be automatically identified.


In [18]:
print("\nPreparing technical failure-mode observations")

failure_modes = [
    {
        "Failure_Mode":"High prediction error cases",
        "Possible_Reason":"Certain operating conditions or charging phases may be less represented or more difficult for the model.",
        "Evidence":"High-error samples identified from the existing prediction/error evidence.",
        "Action": "Monitor these cases during future validation and testing."
    },
    {
        "Failure_Mode":"Charging-phase transitions",
        "Possible_Reason":"Changes between charging states can create rapidly changing electrical characteristics.",
        "Evidence":"Battery voltage, current, temperature and accumulated energy can change across charging phases.",
        "Action":"Treat phase-transition cases as potential sources of larger prediction error."
    },
    {
        "Failure_Mode":"Temperature variation",
        "Possible_Reason":"Battery behaviour changes with operating temperature.",
        "Evidence":"Temperature is one of the final model input features.",
        "Action":"Continue checking model behaviour across different temperature conditions."
    },
    {
        "Failure_Mode":"Cycle-to-cycle variation",
        "Possible_Reason":"Battery characteristics can change between charging cycles.",
        "Evidence":"Cycle is included as a final model feature.",
        "Action":"Check performance across different cycles during future validation."
    },
    {
        "Failure_Mode": "Remaining-time sensitivity",
        "Possible_Reason": "Remaining charging time depends strongly on the current charging state and elapsed charging process.",
        "Evidence":"Elapsed Time and electrical measurements are used by the final Remaining Charging Time model.",
        "Action":"Monitor predictions near charging-phase changes and session boundaries."
    }
]
failure_modes_df = pd.DataFrame(failure_modes)
print(failure_modes_df[["Failure_Mode","Possible_Reason","Action"]].to_string(index=False))


Preparing technical failure-mode observations
               Failure_Mode                                                                                          Possible_Reason                                                                        Action
High prediction error cases Certain operating conditions or charging phases may be less represented or more difficult for the model.                     Monitor these cases during future validation and testing.
 Charging-phase transitions                  Changes between charging states can create rapidly changing electrical characteristics. Treat phase-transition cases as potential sources of larger prediction error.
      Temperature variation                                                    Battery behaviour changes with operating temperature.    Continue checking model behaviour across different temperature conditions.
   Cycle-to-cycle variation                                              Battery characteristics can change b

In [19]:
print("\n Saving Day 25 analysis files")
ERROR_STATS_PATH = os.path.join( DAY25_PATH,"DAY25_ERROR_STATISTICS.csv")
FAILURE_MODES_PATH = os.path.join(DAY25_PATH,"DAY25_FAILURE_MODE_ANALYSIS.csv")
PREDICTION_ANALYSIS_PATH = os.path.join(DAY25_PATH,"DAY25_PREDICTION_ERROR_ANALYSIS.csv")
if error_statistics:
    error_statistics_df.to_csv(ERROR_STATS_PATH,index=False)
    print("Saved:", ERROR_STATS_PATH)
if not prediction_analysis_df.empty:
    prediction_analysis_df.to_csv(PREDICTION_ANALYSIS_PATH,index=False)
    print(" Saved:",PREDICTION_ANALYSIS_PATH)
failure_modes_df.to_csv(FAILURE_MODES_PATH,index=False)
print(" Saved:",FAILURE_MODES_PATH)


 Saving Day 25 analysis files
Saved: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day25_outputs/DAY25_ERROR_STATISTICS.csv
 Saved: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day25_outputs/DAY25_FAILURE_MODE_ANALYSIS.csv


In [20]:
DAY25_RECORD_PATH = os.path.join(DAY25_PATH,"DAY25_ERROR_FAILURE_REVIEW_RECORD.json")
day25_record = {
    "day": 25,
    "title":"Detailed Error and Failure Analysis",
    "purpose":"Document error behaviour and potential failure modes using existing validation and prediction evidence.",
    "source":"Existing Day 20 analysis and frozen project evidence.",
    "retraining_performed": False,
    "new_model_comparison_performed":False,
    "models_modified":False,
    "new_predictions_required":False,
    "prediction_analysis_available":not prediction_analysis_df.empty,
    "error_statistics_available":bool(error_statistics),
    "failure_mode_categories":[item["Failure_Mode"]for item in failure_modes]}
with open(
    DAY25_RECORD_PATH,
    "w"
) as f:
    json.dump(day25_record,f,indent=4)
print("Saved:",DAY25_RECORD_PATH)

Saved: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day25_outputs/DAY25_ERROR_FAILURE_REVIEW_RECORD.json


In [21]:
DAY25_NOTE_PATH = os.path.join(DAY25_PATH,"DAY25_SUBMISSION_NOTE.txt")
with open(
    DAY25_NOTE_PATH,
    "w"
) as f:
    f.write(
        "DAY 25 — DETAILED ERROR / FAILURE ANALYSIS\n"
        "The existing validation and prediction evidence was reviewed\n"
        "to document error behaviour and potential failure modes.\n\n"
        "Areas considered:\n"
        "- High-error prediction cases\n"
        "- Charging-phase transitions\n"
        "- Temperature variation\n"
        "- Cycle-to-cycle variation\n"
        "- Remaining charging-time sensitivity\n\n"
        "No model retraining was performed.\n"
        "No new model comparison was performed.\n"
        "No frozen model was modified.\n"
        "The analysis builds on the existing validation evidence.\n"
    )
print("Saved:",DAY25_NOTE_PATH)


Saved: /content/drive/MyDrive/BATTERY_SOC_PROJECT/day25_outputs/DAY25_SUBMISSION_NOTE.txt


In [22]:
print("\nDay 25 outputs:")
print(" • DAY25_ERROR_STATISTICS.csv")
print(" • DAY25_PREDICTION_ERROR_ANALYSIS.csv (if detected)")
print(" • DAY25_FAILURE_MODE_ANALYSIS.csv")
print(" • DAY25_ERROR_FAILURE_REVIEW_RECORD.json")
print(" • DAY25_SUBMISSION_NOTE.txt")
print("\nNo retraining was performed.")
print("No frozen model was modified.")


Day 25 outputs:
 • DAY25_ERROR_STATISTICS.csv
 • DAY25_PREDICTION_ERROR_ANALYSIS.csv (if detected)
 • DAY25_FAILURE_MODE_ANALYSIS.csv
 • DAY25_ERROR_FAILURE_REVIEW_RECORD.json
 • DAY25_SUBMISSION_NOTE.txt

No retraining was performed.
No frozen model was modified.
